# MS3 — Step 6: PRODIGY Binding Affinity

Runs PRODIGY (BonvinLab/WeNMR) on each of the 35 AF3-predicted structures to estimate
binding free energy (ΔG) and dissociation constant (Kd) from interface contact statistics.
This replaces Boltz-2's affinity head, which was never validated for protein-protein
interactions (only protein-ligand) -- PRODIGY is the scientifically appropriate tool for
HLA-peptide complexes specifically.

**Chain selection follows the same locus-aware logic verified in Step 5**: class I pairs
use HLA heavy chain vs. peptide (B2M excluded, confirmed non-contacting); class II pairs
use DRA+DRB1 jointly vs. peptide (confirmed both chains genuinely contact the peptide in
solved class II structures).

**Compute stack:** prodigy-prot (pip-installable, verified working directly against a
real structure before this notebook was written -- confirmed producing genuine ΔG/Kd
output, not just installing cleanly).

**Workflow:**
1. Load the 35 structure-ready pairs (Part A)
2. Install and verify PRODIGY (Part B)
3. Run PRODIGY on all 35 structures with correct chain selection (Part C)
4. Merge with Step 5's groove geometry table and export (Part D)

## Part A — Load Structure-Ready Pairs

In [ ]:
!pip install -q pandas 2>/dev/null || pip install -q --break-system-packages pandas

from pathlib import Path
from typing import Optional
import re
import subprocess

import pandas as pd

BASE_DIR = Path("/content/ms3_step6_workspace")
STEP4_DIR = Path("/content/ms3_step4")
RESULTS_SOURCE_DIR = STEP4_DIR / "af3_results"
STEP4_MANIFEST_SOURCE = STEP4_DIR / "step4_final_manifest.csv"
TABLES_DIR = BASE_DIR / "tables"

for d in (BASE_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

MANIFEST_SOURCE = BASE_DIR / "step4_final_manifest.csv"
if STEP4_MANIFEST_SOURCE.exists() and not MANIFEST_SOURCE.exists():
    import shutil
    shutil.copy(STEP4_MANIFEST_SOURCE, MANIFEST_SOURCE)
    print(f"Found Step 4's manifest already in this session -- copied to {MANIFEST_SOURCE}")

if not MANIFEST_SOURCE.exists():
    try:
        from google.colab import files
        print("Upload step4_final_manifest.csv:")
        uploaded = files.upload()
        for fname, content in uploaded.items():
            (BASE_DIR / fname).write_bytes(content)
        candidates = list(BASE_DIR.glob("*manifest*.csv"))
        if candidates and not MANIFEST_SOURCE.exists():
            candidates[0].rename(MANIFEST_SOURCE)
    except ImportError:
        pass

if not MANIFEST_SOURCE.exists():
    raise SystemExit(f"[BLOCKED] {MANIFEST_SOURCE} not found.")

full_manifest = pd.read_csv(MANIFEST_SOURCE)
ready_pairs = full_manifest[full_manifest["status"] == "structure_ready"].copy()

print(f"Structure-ready pairs: {len(ready_pairs)}")
print(ready_pairs["locus"].value_counts().to_string())

if not RESULTS_SOURCE_DIR.exists():
    print(f"\n[WARNING] {RESULTS_SOURCE_DIR} not found in this session -- "
          f"sync structures from Drive before continuing (same pattern as Step 5).")

ready_pairs.head(10)

## Part B — Install and Verify PRODIGY

`prodigy-prot` is pip-installable and was confirmed directly (before this notebook was
written) to correctly parse a real structure and produce genuine ΔG/Kd output -- not
just import cleanly.

In [ ]:
!pip install -q prodigy-prot 2>/dev/null || pip install -q --break-system-packages prodigy-prot

import subprocess

result = subprocess.run(["prodigy", "--help"], capture_output=True, text=True)
if result.returncode == 0 or "usage" in result.stdout.lower() or "usage" in result.stderr.lower():
    print("PRODIGY CLI verified available.")
else:
    print("[WARNING] PRODIGY CLI may not be correctly installed:")
    print(result.stderr[:500])

## Part C — Run PRODIGY on All 35 Structures

Uses each pair's locus to select the correct chains: class I -> A (HLA) vs C (peptide);
class II -> A+B (DRA+DRB1) vs C (peptide). PRODIGY's `--selection` flag takes two groups;
for class II, chains A and B are passed together as the first group.

In [ ]:
def find_structure_file(pair_id: str) -> Optional[Path]:
    folder = RESULTS_SOURCE_DIR / pair_id
    if not folder.exists():
        return None
    cifs = list(folder.glob("*model_0.cif"))
    return cifs[0] if cifs else None

def get_prodigy_selection(locus: str) -> tuple:
    """
    Returns (hla_group, peptide_chain) for PRODIGY's --selection flag.
    Matches the chain logic verified in Step 5: class I excludes B2M
    (chain B), class II includes both DRA and DRB1 (chains A+B) since
    both genuinely contact the peptide in solved class II structures.
    """
    if "DRB1" in locus:
        return ("A,B", "C")
    else:
        return ("A", "C")

def run_prodigy(cif_path: Path, locus: str) -> dict:
    hla_group, pep_chain = get_prodigy_selection(locus)
    selection_arg = f"{hla_group}" if "," not in hla_group else hla_group
    cmd = ["prodigy", str(cif_path), "--selection", hla_group, pep_chain]

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    except subprocess.TimeoutExpired:
        return {"status": "timeout", "delta_g_kcal_mol": None, "kd_M": None}

    output = result.stdout + result.stderr
    if result.returncode != 0 and "Predicted binding affinity" not in output:
        return {"status": f"failed: {output[-300:]}", "delta_g_kcal_mol": None, "kd_M": None}

    delta_g = None
    kd = None
    for line in output.splitlines():
        if "Predicted binding affinity" in line:
            try:
                delta_g = float(line.split(":")[-1].strip())
            except ValueError:
                pass
        if "Predicted dissociation constant" in line:
            try:
                kd_str = line.split(":")[-1].strip()
                kd = float(kd_str)
            except ValueError:
                pass

    if delta_g is None:
        return {"status": f"no_deltaG_parsed: {output[-300:]}", "delta_g_kcal_mol": None, "kd_M": None}

    return {"status": "ok", "delta_g_kcal_mol": delta_g, "kd_M": kd}

prodigy_rows = []
for _, row in ready_pairs.iterrows():
    pair_id = row["pair_id"]
    cif_path = find_structure_file(pair_id)

    if cif_path is None:
        print(f"  [SKIP] {pair_id}: no structure file found")
        prodigy_rows.append({"pair_id": pair_id, "status": "no_structure_file",
                             "delta_g_kcal_mol": None, "kd_M": None})
        continue

    result = run_prodigy(cif_path, row["locus"])
    result["pair_id"] = pair_id
    result["allele"] = row["allele"]
    result["locus"] = row["locus"]
    result["peptide"] = row["peptide"]
    prodigy_rows.append(result)

    flag = "" if result["status"] == "ok" else f"  [{result['status'][:60]}]"
    print(f"  {pair_id} ({row['allele']}): dG={result.get('delta_g_kcal_mol')} kcal/mol  "
          f"Kd={result.get('kd_M')} M{flag}")

prodigy_df = pd.DataFrame(prodigy_rows)
n_ok = (prodigy_df["status"] == "ok").sum()
print(f"\n{n_ok}/{len(prodigy_df)} pairs successfully scored")

n_positive_dg = (prodigy_df["delta_g_kcal_mol"] > 0).sum()
if n_positive_dg > 0:
    print(f"[NOTE] {n_positive_dg} pair(s) show a positive (unfavorable) predicted "
          f"binding affinity -- for real HLA-peptide complexes this is unexpected and "
          f"worth inspecting those specific structures directly, since a genuine "
          f"groove-bound peptide should show favorable (negative) binding energy.")

prodigy_df

## Part D — Merge with Step 5's Groove Geometry and Export

In [ ]:
STEP5_TABLE = Path("/content/ms3_step5_workspace/tables/Table_S2_groove_geometry_concordance.csv")

if STEP5_TABLE.exists():
    step5_df = pd.read_csv(STEP5_TABLE)
    final_df = step5_df.merge(
        prodigy_df[["pair_id", "delta_g_kcal_mol", "kd_M", "status"]].rename(
            columns={"status": "prodigy_status"}),
        on="pair_id", how="left"
    )
    print(f"Merged with Step 5's table ({len(step5_df)} rows)")
else:
    print(f"[INFO] Step 5's table not found at {STEP5_TABLE} -- exporting PRODIGY results alone.")
    final_df = prodigy_df

out_path = TABLES_DIR / "Table_S3_prodigy_affinity.csv"
final_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

try:
    from google.colab import files
    files.download(str(out_path))
except ImportError:
    print(f"Table saved locally at: {out_path}")

print("\nStep 6 complete. Feed this table into Step 7 (immunogenicity correlation).")
final_df